# PJM Load Forecast — DA Cutoff Revision Analysis (v3: RTO Only)

Inspects how PJM RTO load forecasts for the next 7 days evolve leading up to the DA market window.
The last forecast before 9am EST is the cutoff vintage a trader uses for DA bidding.
We compare the cutoff forecast to 12h / 24h / 48h earlier vintages to reveal
revision direction, magnitude, and patterns across forward dates.

## 1. Setup & Data Pull

In [40]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "backend"))
from src.utils.azure_postgresql import pull_from_db

In [41]:
# Load the main cutoff analysis query (v3: RTO only, forward dates)
sql_path = Path.cwd() / "pjm_load_forecast_da_cutoff_v3.sql"
query = sql_path.read_text()
df = pull_from_db(query=query)

print(f"{len(df):,} rows")
print(f"Date range: {df['forecast_date'].min()} to {df['forecast_date'].max()}")
df.head(10)

144 rows
Date range: 2026-03-04 to 2026-03-10


,forecast_date,hour_ending,region,cutoff_load_mw,cutoff_execution_dt,exec_dt_12h,load_mw_12h,exec_dt_24h,load_mw_24h,exec_dt_48h,load_mw_48h,delta_12h,delta_24h,delta_48h
0,2026-03-10,1,RTO,75055.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
1,2026-03-10,2,RTO,72759.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
2,2026-03-10,3,RTO,71458.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
3,2026-03-10,4,RTO,70972.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
4,2026-03-10,5,RTO,72178.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
5,2026-03-10,6,RTO,75806.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
6,2026-03-10,7,RTO,82193.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
7,2026-03-10,8,RTO,86588.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
8,2026-03-10,9,RTO,87498.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN
9,2026-03-10,10,RTO,86943.0,2026-03-04 08:47:16,NaT,NaN,NaT,NaN,NaT,NaN,NaN,NaN,NaN


## 1b. Forecast Coverage Summary

One row per forecast date showing the cutoff vintage used, each lookback vintage,
and peak cutoff load — a quick reference for which days have full 48h→12h history.

In [42]:
# Build a per-forecast-date summary table
_fmt_dt = lambda s: pd.to_datetime(s).strftime("%Y-%m-%d %H:%M") if pd.notna(s) else "—"

forecast_summary = (
    df.groupby("forecast_date")
    .agg(
        cutoff_exec=("cutoff_execution_dt", "first"),
        exec_12h=("exec_dt_12h", "first"),
        exec_24h=("exec_dt_24h", "first"),
        exec_48h=("exec_dt_48h", "first"),
        peak_cutoff_mw=("cutoff_load_mw", "max"),
        min_cutoff_mw=("cutoff_load_mw", "min"),
        hours_covered=("hour_ending", "nunique"),
    )
    .reset_index()
    .sort_values("forecast_date")
)

# Days from today
forecast_summary["days_ahead"] = (
    pd.to_datetime(forecast_summary["forecast_date"]) - pd.Timestamp.now().normalize()
).dt.days

# Format execution timestamps
for col in ["cutoff_exec", "exec_12h", "exec_24h", "exec_48h"]:
    forecast_summary[col] = forecast_summary[col].apply(_fmt_dt)

# Lookback availability flags
forecast_summary["has_12h"] = forecast_summary["exec_12h"] != "—"
forecast_summary["has_24h"] = forecast_summary["exec_24h"] != "—"
forecast_summary["has_48h"] = forecast_summary["exec_48h"] != "—"

display_cols = [
    "forecast_date", "days_ahead", "hours_covered",
    "cutoff_exec", "exec_48h", "exec_24h", "exec_12h",
    "has_48h", "has_24h", "has_12h",
    "peak_cutoff_mw", "min_cutoff_mw",
]
forecast_summary[display_cols].style.set_caption(
    "Forecast Date Coverage — 7-Day Forward Window"
).format({
    "peak_cutoff_mw": "{:,.0f}",
    "min_cutoff_mw": "{:,.0f}",
}).set_properties(**{"text-align": "center"})

,forecast_date,days_ahead,hours_covered,cutoff_exec,exec_48h,exec_24h,exec_12h,has_48h,has_24h,has_12h,peak_cutoff_mw,min_cutoff_mw
0,2026-03-04,0,24,2026-03-04 08:47,2026-03-02 08:47,2026-03-03 08:17,2026-03-03 20:47,True,True,True,"101,005","85,781"
1,2026-03-05,1,24,2026-03-04 08:47,2026-03-02 08:47,2026-03-03 08:17,2026-03-03 20:47,True,True,True,"96,543","79,141"
2,2026-03-06,2,24,2026-03-04 08:47,2026-03-02 08:47,2026-03-03 08:17,2026-03-03 20:47,True,True,True,"94,428","75,921"
3,2026-03-07,3,24,2026-03-04 08:47,2026-03-02 08:47,2026-03-03 08:17,2026-03-03 20:47,True,True,True,"88,122","72,747"
4,2026-03-09,5,24,2026-03-04 08:47,—,2026-03-03 08:17,2026-03-03 20:47,False,True,True,"92,337","70,751"
5,2026-03-10,6,24,2026-03-04 08:47,—,—,—,False,False,False,"96,021","70,972"


## 2. Data Validation — Cutoff Vintage Inspection

In [43]:
# Actual cutoff execution timestamps per forecast_date
cutoff_times = (
    df.groupby("forecast_date")["cutoff_execution_dt"]
    .first()
    .reset_index()
)
cutoff_times["cutoff_hour"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.hour
cutoff_times["cutoff_minute"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.minute
cutoff_times["cutoff_time_str"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.strftime("%H:%M")

print("Cutoff execution timestamps (should all be before 09:00 EST):")
cutoff_times[["forecast_date", "cutoff_execution_dt", "cutoff_time_str"]]

Cutoff execution timestamps (should all be before 09:00 EST):


,forecast_date,cutoff_execution_dt,cutoff_time_str
0,2026-03-04,2026-03-04 08:47:16,08:47
1,2026-03-05,2026-03-04 08:47:16,08:47
2,2026-03-06,2026-03-04 08:47:16,08:47
3,2026-03-07,2026-03-04 08:47:16,08:47
4,2026-03-09,2026-03-04 08:47:16,08:47
5,2026-03-10,2026-03-04 08:47:16,08:47


In [44]:
# Bar chart of cutoff hour-of-day
fig = px.bar(
    cutoff_times,
    x="forecast_date",
    y="cutoff_hour",
    text="cutoff_time_str",
    title="RTO — Cutoff Vintage Hour-of-Day (EST) by Forecast Date",
    template="plotly_dark",
)
fig.add_hline(y=9, line_dash="dash", line_color="red", annotation_text="9am EST cutoff")
fig.update_layout(yaxis_title="Hour (EST)", xaxis_title="Forecast Date")
fig.show()

In [45]:
# Lookback coverage: which dates have 12h/24h/48h data
coverage = (
    df.groupby("forecast_date")
    .agg(
        has_12h=("load_mw_12h", lambda x: x.notna().any()),
        has_24h=("load_mw_24h", lambda x: x.notna().any()),
        has_48h=("load_mw_48h", lambda x: x.notna().any()),
        exec_dt_12h=("exec_dt_12h", "first"),
        exec_dt_24h=("exec_dt_24h", "first"),
        exec_dt_48h=("exec_dt_48h", "first"),
    )
    .reset_index()
)
print("Lookback coverage:")
coverage

Lookback coverage:


,forecast_date,has_12h,has_24h,has_48h,exec_dt_12h,exec_dt_24h,exec_dt_48h
0,2026-03-04,True,True,True,2026-03-03 20:47:09,2026-03-03 08:17:32,2026-03-02 08:47:09
1,2026-03-05,True,True,True,2026-03-03 20:47:09,2026-03-03 08:17:32,2026-03-02 08:47:09
2,2026-03-06,True,True,True,2026-03-03 20:47:09,2026-03-03 08:17:32,2026-03-02 08:47:09
3,2026-03-07,True,True,True,2026-03-03 20:47:09,2026-03-03 08:17:32,2026-03-02 08:47:09
4,2026-03-09,True,True,False,2026-03-03 20:47:09,2026-03-03 08:17:32,NaT
5,2026-03-10,False,False,False,NaT,NaT,NaT


## 3. Forecast Evolution — Cutoff vs Lookbacks

In [46]:
# Overlay 48h -> 24h -> 12h -> cutoff for the latest forecast_date
latest_date = df["forecast_date"].max()
df_latest = df[df["forecast_date"] == latest_date].copy().sort_values("hour_ending")

colors = {"48h": "#636EFA", "24h": "#EF553B", "12h": "#00CC96", "Cutoff": "#FFA15A"}
dashes = {"48h": "dot", "24h": "dash", "12h": "dashdot", "Cutoff": "solid"}

fig = go.Figure()
for label, col, width in [
    ("48h", "load_mw_48h", 1),
    ("24h", "load_mw_24h", 1.5),
    ("12h", "load_mw_12h", 2),
    ("Cutoff", "cutoff_load_mw", 3),
]:
    fig.add_trace(
        go.Scatter(
            x=df_latest["hour_ending"],
            y=df_latest[col],
            name=label,
            line=dict(color=colors[label], dash=dashes[label], width=width),
        )
    )

fig.update_layout(
    height=450,
    template="plotly_dark",
    title=f"RTO — Forecast Evolution — Latest Date ({latest_date})",
    xaxis_title="Hour Ending",
    yaxis_title="Load (MW)",
)
fig.show()

In [47]:
# All dates — cutoff (solid) vs 24h lookback (dashed)
dates = sorted(df["forecast_date"].unique())

fig = go.Figure()
for d in dates:
    ddf = df[df["forecast_date"] == d].sort_values("hour_ending")
    opacity = 0.3 if d != latest_date else 1.0
    # Cutoff
    fig.add_trace(go.Scatter(
        x=ddf["hour_ending"], y=ddf["cutoff_load_mw"],
        name=f"{d} cutoff",
        line=dict(width=2 if d == latest_date else 1),
        opacity=opacity,
        showlegend=True,
    ))
    # 24h lookback (dashed)
    fig.add_trace(go.Scatter(
        x=ddf["hour_ending"], y=ddf["load_mw_24h"],
        name=f"{d} 24h",
        line=dict(dash="dash", width=1),
        opacity=opacity * 0.6,
        showlegend=False,
    ))

fig.update_layout(
    height=500, template="plotly_dark",
    title="RTO — Cutoff (solid) vs 24h Prior (dashed)",
    xaxis_title="Hour Ending", yaxis_title="Load (MW)",
)
fig.show()

## 4. Forecast Deltas — MW Changes at Each Lookback

In [48]:
# Grouped bar chart: delta_12h, delta_24h, delta_48h by hour_ending for latest date
fig = go.Figure()
for col, label, color in [
    ("delta_48h", "48h\u2192Cutoff", "#636EFA"),
    ("delta_24h", "24h\u2192Cutoff", "#EF553B"),
    ("delta_12h", "12h\u2192Cutoff", "#00CC96"),
]:
    fig.add_trace(go.Bar(
        x=df_latest["hour_ending"], y=df_latest[col],
        name=label, marker_color=color,
    ))

fig.update_layout(
    barmode="group",
    title=f"RTO — Forecast Revision Deltas ({latest_date})",
    xaxis_title="Hour Ending", yaxis_title="Delta (MW)",
    template="plotly_dark", height=400,
)
fig.add_hline(y=0, line_color="white", line_width=0.5)
fig.show()

In [49]:
# Heatmap: 24h delta across all dates x hours
pivot_24h = df.pivot_table(
    index="forecast_date", columns="hour_ending", values="delta_24h",
)

fig = px.imshow(
    pivot_24h.values,
    x=[f"HE{int(c)}" for c in pivot_24h.columns],
    y=[str(d) for d in pivot_24h.index],
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    title="RTO — 24h Forecast Revision (MW) by Date \u00d7 Hour",
    labels={"color": "Delta MW"},
    template="plotly_dark",
)
fig.update_layout(height=500)
fig.show()

## 5. Per-Date Revision Deep Dive

For each forecast date, show the 48h → 24h → 12h → cutoff evolution
so we can inspect how each day's forecast converged.

In [50]:
# One subplot row per date
dates = sorted(df["forecast_date"].unique())

fig = make_subplots(
    rows=len(dates), cols=1,
    shared_xaxes=True,
    subplot_titles=[f"RTO — {d}" for d in dates],
    vertical_spacing=0.04,
)

for i, d in enumerate(dates, 1):
    ddf = df[df["forecast_date"] == d].sort_values("hour_ending")

    for label, col, exec_col, width in [
        ("48h", "load_mw_48h", "exec_dt_48h", 1),
        ("24h", "load_mw_24h", "exec_dt_24h", 1.5),
        ("12h", "load_mw_12h", "exec_dt_12h", 2),
        ("Cutoff", "cutoff_load_mw", "cutoff_execution_dt", 3),
    ]:
        # Build customdata: [exec_dt, forecast_date, delta_from_cutoff, rank_label]
        exec_dts = ddf[exec_col].astype(str).values
        forecast_dates = ddf["forecast_date"].astype(str).values
        # Delta from cutoff (cutoff has 0 delta)
        if label == "Cutoff":
            deltas = np.zeros(len(ddf))
        else:
            delta_col_name = f"delta_{label}"  # e.g. delta_12h, delta_24h, delta_48h
            deltas = ddf[delta_col_name].values
        # Rank label for ordering context
        rank_labels = np.array([label] * len(ddf))

        customdata = np.column_stack([exec_dts, forecast_dates, deltas, rank_labels])

        fig.add_trace(
            go.Scatter(
                x=ddf["hour_ending"],
                y=ddf[col],
                name=label,
                line=dict(color=colors[label], dash=dashes[label], width=width),
                showlegend=(i == 1),
                customdata=customdata,
                hovertemplate=(
                    "<b>%{customdata[3]}</b><br>"
                    "Forecast Date: %{customdata[1]}<br>"
                    "HE: %{x}<br>"
                    "Load: %{y:,.0f} MW<br>"
                    "Execution DT: %{customdata[0]}<br>"
                    "Delta to Cutoff: %{customdata[2]} MW"
                    "<extra></extra>"
                ),
            ),
            row=i, col=1,
        )

fig.update_layout(
    height=250 * len(dates),
    template="plotly_dark",
    title_text="RTO — Forecast Evolution by Date",
)
fig.update_yaxes(title_text="Load (MW)")
fig.update_xaxes(title_text="Hour Ending", row=len(dates), col=1)
fig.show()

## 6. Revision Summary Statistics

In [51]:
# Summary table: mean, median, std of deltas by lookback
summary_rows = []
for lookback, col in [("12h", "delta_12h"), ("24h", "delta_24h"), ("48h", "delta_48h")]:
    vals = df[col].dropna()
    summary_rows.append({
        "Lookback": lookback,
        "Mean (MW)": vals.mean(),
        "Median (MW)": vals.median(),
        "Std (MW)": vals.std(),
        "Min (MW)": vals.min(),
        "Max (MW)": vals.max(),
        "N": len(vals),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Lookback,Mean (MW),Median (MW),Std (MW),Min (MW),Max (MW),N
0,12h,678.950000,718.5,601.249383,-679.0,1770.0,120
1,24h,894.958333,1177.5,881.180915,-1797.0,2406.0,120
2,48h,1653.364583,1721.0,816.532413,-381.0,3321.0,96


In [52]:
# Per-date summary: avg revision direction across all hours
date_summary = (
    df.groupby("forecast_date")
    .agg(
        mean_delta_12h=("delta_12h", "mean"),
        mean_delta_24h=("delta_24h", "mean"),
        mean_delta_48h=("delta_48h", "mean"),
        peak_cutoff_mw=("cutoff_load_mw", "max"),
    )
    .reset_index()
)

# Grouped bar: mean delta per date for each lookback
fig = go.Figure()
for col, label, color in [
    ("mean_delta_48h", "48h", "#636EFA"),
    ("mean_delta_24h", "24h", "#EF553B"),
    ("mean_delta_12h", "12h", "#00CC96"),
]:
    fig.add_trace(go.Bar(
        x=date_summary["forecast_date"], y=date_summary[col],
        name=label, marker_color=color,
    ))

fig.update_layout(
    barmode="group",
    title="RTO \u2014 Mean Forecast Revision (MW) by Date & Lookback",
    xaxis_title="Forecast Date", yaxis_title="Avg Delta (MW)",
    template="plotly_dark", height=450,
)
fig.add_hline(y=0, line_color="white", line_width=0.5)
fig.show()

In [53]:
# Heatmaps for all lookbacks (12h, 24h, 48h)
for lookback, delta_col in [("12h", "delta_12h"), ("24h", "delta_24h"), ("48h", "delta_48h")]:
    pivot = df.pivot_table(
        index="forecast_date", columns="hour_ending", values=delta_col,
    )
    if pivot.empty:
        continue

    fig = px.imshow(
        pivot.values,
        x=[f"HE{int(c)}" for c in pivot.columns],
        y=[str(d) for d in pivot.index],
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        aspect="auto",
        title=f"RTO \u2014 {lookback} Forecast Revision (MW) by Date \u00d7 Hour",
        labels={"color": "Delta MW"},
        template="plotly_dark",
    )
    fig.update_layout(height=400)
    fig.show()